# Questionnaire detail  `[EVAL]`  — family `arms/questionnaires`

**Level 2 of the per-arm drill-down: inside each questionnaire, all four arms on one axis.** One uniform section per rubric, each in the same small-multiples style as `arms/outcomes`' `trajectories_all_metrics`: a per-item / per-component **detail grid** (trajectories across iterations, the four arms PTO/GRPO × K=0/K=5 overlaid) + **"which items drive the change"** delta bars at each arm's **final AND best** iteration + one merged table (`target` column). Likert-item rubrics (Q1, Q2, WAI-SR, CSQ-8, MI-SAT) use the generic item loader; MITI / PCT / MICI decompose into their oracle-annotated behaviour rates + globals (per-metric zooms in the nested groups `miti/`, `pct/`, `mici/`).

**What this family answers.** *Which items / components of each instrument move under training, when, and does the composition differ by optimizer or by look-ahead depth?* It is descriptive and per-arm: no K-contrast statistics live here (those are `lookahead/reward` + `lookahead/behaviour`); no method contrast either (`method/contrast`). Δ vs base is a difference of arm means over the same 96 personas (base = that arm's own iteration-0 draw), not a persona-paired test.

**Per-judge family.** Rendered once per grader on disk → `results/arms/questionnaires/{figures,tables}/<judge>/`. `EDA_JUDGE=""` = the primary oracle (gpt-4o-mini, which was also the training reward); a judge tag reads that held-out grader's partition of the score lake instead. Never average across the two.

**Support.** Every trajectory panel ends at that arm's last SCORED state, and how far that reaches can differ by grader, so the exact iteration is derived per render and printed in the support line of every caption — which says so explicitly when, as now, all four arms reach the same one.

Global scores → `arms/outcomes`; the validity / reward-hacking synthesis these details feed → `arms/validity`; per-arm stats → `arms/stats`.

In [ ]:
import sys, os
_p = os.path.abspath(".")                      # find eda/ (the dir holding eda_analysis/) from any depth
while _p != os.path.dirname(_p) and not os.path.isdir(os.path.join(_p, "eda_analysis")):
    _p = os.path.dirname(_p)
sys.path.insert(0, _p)
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.width", 185, "display.max_columns", 50)
import eda_analysis
from eda_analysis import exports, plotting, stats, behavior
from eda_analysis.constants import judge_dirname, arm_label

cfg = eda_analysis.EdaConfig(family="arms/questionnaires", judge=os.environ.get("EDA_JUDGE", ""))
S = eda_analysis.notebook_setup(cfg)
exports.reset_results()   # clears only THIS family's generated figures/tables for the active judge (never SUMMARY.md)
exports.save_provenance(cfg, S.SCORES)   # re-stamp the leaf's _provenance.md (reset just removed the one notebook_setup wrote)

# Best iteration per arm (own training oracle — the checkpoint you'd select); drives the *_best deltas.
BEST = eda_analysis.best_iteration_by_arm(S.SCORES)
print("best iteration per arm:", BEST)

# ── caption building blocks (every save_* below states grader, unit, sign, censoring) ────────
JLABEL = judge_dirname(S.JUDGE)
GRADER = (f"grader: {JLABEL} (primary oracle = the training reward)" if not S.JUDGE
          else f"grader: {JLABEL} (held-out judge, not the training reward)")
# Derived from S.SCORES - the SCORES this leaf plots - NOT from S.ARMS: Arm.iters lists the
# conversation dirs on disk, which can run FURTHER than the score lake, so the S.ARMS version of this
# caption named an iteration no trajectory panel in this notebook ever reaches.
CENSOR = (eda_analysis.support_note(S.SCORES, subject=f"no later state scored by {JLABEL}")
          or f"All arms run to the same last scored iteration under {JLABEL}.")
ARMS_TXT = "four arms (PTO/GRPO x K=0/K=5) overlaid"
DELTA_UNIT = ("Delta = target-iteration mean minus that arm's own iteration-0 (base) mean, a difference of "
              "arm means over the same 96 personas (not a persona-paired contrast)")
print(GRADER); print(CENSOR)

## 0 · Section helpers
The uniform Likert-item pattern (Q1 / Q2 / WAI-SR / CSQ-8 / MI-SAT): detail grid + final/best delta bars + one merged table. Q2 keeps its face-content group colours (`q2_style`). The behaviour-frame pattern (MITI / PCT / MICI) melts the by-iter frame into `(item, score)` rows so the same `stats.item_endpoint_deltas` drives the final/best delta bars + merged table.

In [ ]:
def item_section(qname, slug, q2_style=False):
    IT = eda_analysis.load_items(qname, S.ARMS)
    if IT.empty:
        print(f"{qname} not scored yet — run Run_Eval.ipynb.")
        return IT
    lab = eda_analysis.display_label(qname)
    fig = plotting.item_trajectory_grid(IT, palette=S.PALETTE, title=f"{lab} — per-item trajectories")
    if fig:
        exports.save_fig(fig, f"{slug}_detail_grid",
                         caption=f"{qname}: per-item mean +/- 95% CI (bootstrap over conversations) across "
                                 f"iterations, {ARMS_TXT} — the item-level drill-down behind the {qname} "
                                 f"global score. Items on the 1-5 Likert scale, higher = better. {GRADER}. {CENSOR}")
        plt.show()
    deltas = {}
    for tgt, note, tmap in (("final", "final iteration", None),
                            ("best", "best iteration (own oracle)", BEST)):
        D = (stats.q2_item_endpoint_deltas(IT, target_iter_by_arm=tmap) if q2_style
             else stats.item_endpoint_deltas(IT, target_iter_by_arm=tmap))
        deltas[tgt] = D
        figd = (plotting.q2_item_delta_bars(D) if q2_style
                else plotting.item_delta_bars(D, title=f"{qname} — which items drive the change ({note})",
                                              xlabel="Δ item mean vs base (1–5 scale)"))
        if figd:
            exports.save_fig(figd, f"{slug}_item_deltas_{tgt}",
                             caption=f"Delta vs base per {qname} item at each arm's {note} — one panel per arm "
                                     f"(four arms), shared Delta scale, panel title = the arm's target iteration. "
                                     f"{DELTA_UNIT}; + = higher item score (1-5 Likert). "
                                     + ("Bars coloured by our face-content item group (analytical, not a validated "
                                        "subscale). " if q2_style else "")
                                     + f"{GRADER}. {CENSOR}")
            plt.show()
    T = pd.concat([deltas["final"].assign(target="final"), deltas["best"].assign(target="best")],
                  ignore_index=True)
    T = T[["target"] + [c for c in T.columns if c != "target"]]
    exports.save_table(T, f"{slug}_item_deltas",
                       caption=f"Per (arm, {qname} item): base / target means + Delta at the final AND best "
                               f"iteration (target column), all four arms. {DELTA_UNIT}; + = higher item score. "
                               f"{GRADER}. {CENSOR}")
    return IT


def detail_deltas(det, slug, qname, xlabel, *, sign_note):
    melted = det.melt(id_vars=["arm", "method", "K", "iteration"],
                      var_name="item", value_name="score").dropna(subset=["score"])
    melted["is_base"] = melted["iteration"].eq(0)
    Tf = None
    for tgt, note, tmap in (("final", "final iteration", None),
                            ("best", "best iteration (own oracle)", BEST)):
        D = stats.item_endpoint_deltas(melted, target_iter_by_arm=tmap,
                                       short=eda_analysis.display_label)
        figd = plotting.item_delta_bars(D, title=f"{qname} — which components drive the change ({note})",
                                        xlabel=xlabel)
        if figd:
            exports.save_fig(figd, f"{slug}_item_deltas_{tgt}",
                             caption=f"Delta vs base per {qname} component at each arm's {note} — one panel per "
                                     f"arm (four arms), shared Delta scale, mixed units (see xlabel). "
                                     f"{DELTA_UNIT}; {sign_note} {GRADER}. {CENSOR}")
            plt.show()
        if tgt == "final":
            Tf = D
        else:
            T = pd.concat([Tf.assign(target="final"), D.assign(target="best")], ignore_index=True)
            T = T[["target"] + [c for c in T.columns if c != "target"]]
            exports.save_table(T, f"{slug}_item_deltas",
                               caption=f"Per (arm, {qname} component): base / target values + Delta at the final "
                                       f"AND best iteration (target column), all four arms. {DELTA_UNIT}; "
                                       f"{sign_note} {GRADER}. {CENSOR}")

## 1 · Q1 — Session Satisfaction (5 items)  `[EVAL]`
Half of the training reward (under the primary oracle). The 5 items separate *chat/content satisfaction* from *motivation facilitated* and *learning (relevance)* — the item view shows whether an arm's reward gain is satisfaction-flavoured or learning-flavoured, and whether the K=5 arms move the same items as their K=0 siblings.

In [ ]:
Q1I = item_section("Q1", "q1")

## 2 · Q2 — Working Alliance / Relational Communication (17 items)  `[EVAL]`
The other half of the training reward, and the reward-composition story: items 1/2/3/10 reward therapist **self-disclosure** — behaviour MI does not prescribe. Bars are coloured by our face-content groups (analytical, not a validated subscale); the group-trajectory figure shows *when* each component takes off in each of the four arms.

In [ ]:
Q2I = item_section("Q2", "q2", q2_style=True)
if not Q2I.empty:
    fig = plotting.q2_item_group_trajectory(Q2I)
    if fig:
        exports.save_fig(fig, "q2_item_group_trajectories",
                         caption="Mean Q2 item-group score across iterations, one panel per arm (four arms; +/-95% CI "
                                 "over conversations x items; face-content groups — analytical, not a validated "
                                 "subscale) — when each reward component takes off, per arm. 1-5 Likert, higher = "
                                 f"better. {GRADER}. {CENSOR}")
        plt.show()

## 3 · WAI-SR — Working Alliance (3 subscales + 12 items)  `[EVAL]`
Held-out alliance instrument (overlaps Q2 by design — both alliance measures). First the validated **Goal / Task / Bond** subscales, then the full 12-item drill-down.

In [ ]:
SUB = eda_analysis.load_subscales(S.ARMS)
fig = plotting.subscale_trajectory_grid(SUB, parents=("WAI-SR",), min_iters=3)
if fig:
    exports.save_fig(fig, "wai_subscales",
                     caption="WAI-SR Goal / Task / Bond subscale means across iterations; one panel per arm (four "
                             "arms; arms with <3 scored iterations omitted). 1-5 Likert, higher = better. "
                             f"{GRADER}. {CENSOR}")
    plt.show()
WAII = item_section("WAI-SR", "wai")

## 4 · CSQ-8 — Client Satisfaction (8 items)  `[EVAL]`
Held-out satisfaction instrument — the item view separates service-quality items from the recommend/return intentions.

In [ ]:
CSQI = item_section("CSQ-8", "csq")

## 5 · MI-SAT — MI Intervention Satisfaction (6 items)  `[EVAL]`
Held-out MI-specific satisfaction — `LikelyChange` (item 6) is the closest thing to a patient-outcome item in the halo cluster.

In [ ]:
MISATI = item_section("MI-SAT", "misat")

## 6 · MITI — MI Treatment Integrity (4 globals + 7 behaviour rates + ratios + official thresholds)  `[EVAL]`
The technique instrument. The detail grid is the questionnaire-level home of the *behaviour drift* figure: the 4 global ratings (1–5), all 7 behaviour counts **per therapist turn** (length-normalized), and the derived proficiency ratios `R:Q` / `%CR` / `%MICO`, four arms overlaid. **Read:** `B6_AF` (affirmations) up while `B3_Q` (questions) down = the affirmation/advice drift — compare its size and timing across the four arms. §6b anchors the summary scores against the **official MITI 4.2.1 competency thresholds** (expert opinion, ~20-min human sessions — out-of-domain caveat applies). Per-metric zooms → group `miti/`.

In [ ]:
MITI_DET = behavior.miti_detail_by_iter(S.ARMS)
if MITI_DET.empty:
    print("MITI not scored yet — run Run_Eval.ipynb to populate this section.")
else:
    MITI_METRICS = [c for c in MITI_DET.columns if c not in ("arm", "method", "K", "iteration")]
    fig = plotting.behavior_trajectory_grid(
        MITI_DET, palette=S.PALETTE, metrics=MITI_METRICS, ncols=4,
        title="MITI detail — global ratings (1–5) + behaviour rates per therapist turn + proficiency ratios")
    if fig:
        exports.save_fig(fig, "miti_detail_grid",
                         caption="MITI drill-down: 4 global ratings (1-5), all 7 behaviour counts PER THERAPIST TURN "
                                 f"(length-normalized) and the derived proficiency ratios R:Q / %CR / %MICO across "
                                 f"iterations, {ARMS_TXT} (arm means per iteration). B6_AF up while B3_Q down = the "
                                 f"affirmation/advice drift; globals higher = better, rates are counts/turn. "
                                 f"{GRADER}. {CENSOR}")
        plt.show()
    # Per-metric zoom (rates + RtoQ + Empathy) -> group miti/.
    for m in [c for c in MITI_METRICS if c.endswith("_per_turn")] + ["RtoQ", "Empathy"]:
        figm = plotting.single_behavior_trajectory(MITI_DET, m, palette=S.PALETTE)
        if figm is not None:
            exports.save_fig(figm, m, group="miti",
                             caption=f"{eda_analysis.display_label(m)} across iterations, {ARMS_TXT} — per-metric "
                                     f"zoom of miti_detail_grid. {GRADER}. {CENSOR}")
            plt.close(figm)
    MT = MITI_DET.round(3).sort_values(["arm", "iteration"]).reset_index(drop=True)
    display(MT)
    exports.save_table(MT, "miti_detail_by_iter",
                       caption="Mean MITI globals (1-5) + behaviour rates per therapist turn + proficiency ratios "
                               f"per (arm, iteration), all four arms. {GRADER}. {CENSOR}")
    detail_deltas(MITI_DET, "miti", "MITI",
                  xlabel="Δ vs base (mixed units: globals 1–5 · rates /turn · ratios)",
                  sign_note="+ = higher value (globals: better; rates/ratios: more of that behaviour).")

### 6b · Official MITI 4.2.1 competency thresholds — absolute anchor  `[EVAL]`
Everything else in the EDA is *relative* (vs base, vs the other arm). This anchors each of the four therapists against the **official MITI 4.2.1 standards** (Moyers, Manuel & Ernst 2014, manual rev. 2015, §I): `R:Q` (fair ≥ 1:1, good ≥ 2:1), `%CR` (≥ 40% / 50%), **Technical global** (≥ 3 / 4), **Relational global** (≥ 3.5 / 4). **Read the verdict table per arm** — which arms cross "fair" / "good" on the global ratings, and whether any reaches "good" on the technique ratios; an `R:Q` rise that coincides with a falling `B3_Q/turn` panel above is a shrinking denominator, not more reflection (cross-check `arms/validity`). **Caveats:** thresholds are expert opinion without normative validation, defined for ~20-min human audio sessions — use as an anchor, not a certification.

In [ ]:
PROF = behavior.miti_proficiency_by_iter(S.ARMS)
fig = plotting.miti_threshold_panel(PROF, palette=S.PALETTE)
if fig is not None:
    exports.save_fig(fig, "miti_proficiency_thresholds",
                     caption="The 4 official MITI 4.2.1 summary scores (R:Q, %CR, Technical global, Relational global) "
                             f"across iterations, {ARMS_TXT}, vs the manual's fair / good competency thresholds "
                             f"(dashed). Higher = more proficient; the per-arm base-vs-final verdicts are in "
                             f"miti_threshold_verdicts. Thresholds are expert opinion (MITI 4.2.1 manual) defined for "
                             f"~20-min human sessions (short text chats are out-of-domain). {GRADER}. {CENSOR}")
    plt.show()
    PROF_T = plotting.miti_threshold_table(PROF)
    display(PROF_T)
    exports.save_table(PROF_T, "miti_threshold_verdicts",
                       caption="Base vs final MITI 4.2.1 summary scores per arm (four arms; 'final' = that arm's last "
                               "scored iteration) with the manual's competency verdict per score (check good / check "
                               f"fair / cross = below basic competence). {GRADER}. {CENSOR}")
else:
    print("MITI not scored yet — run Run_Eval.ipynb to populate the proficiency panel.")

## 7 · PCT — Patient Change-Talk (3 patient globals + talk proportions)  `[EVAL]`
The actual MI *target*, scored from the patient side: Importance / Confidence / Readiness (1–5) + the change/sustain/neutral proportions of patient utterances (`% Sustain` lower = better). **Read:** how far patient motivation moves per arm, and whether the K=5 arm of each optimizer tracks its K=0 sibling. Per-signal zooms → group `pct/`.

In [ ]:
PCT_BEH = behavior.pct_behavior_by_iter(S.ARMS)
if PCT_BEH.empty:
    print("PCT not scored yet — run Run_Eval.ipynb with QUESTIONNAIRE_FILTER=['PCT'] to populate this.")
else:
    PCT_GLOB = [c for c in ["PCT_Importance", "PCT_Confidence", "PCT_Readiness"] if c in PCT_BEH.columns]
    PCT_PROP = [c for c in PCT_BEH.columns if c.endswith("_prop")]
    PCT_METRICS = PCT_GLOB + ["PCT_ChangeProp"] + PCT_PROP
    fig = plotting.behavior_trajectory_grid(
        PCT_BEH, palette=S.PALETTE, metrics=PCT_METRICS,
        title="PCT detail — patient globals (1–5) + patient-utterance proportions")
    if fig:
        exports.save_fig(fig, "pct_detail_grid",
                         caption="PCT patient-perspective globals (Importance / Confidence / Readiness, 1-5, higher = "
                                 "better) + change / sustain / neutral proportions of patient utterances (0-1; % Sustain "
                                 f"lower = better) across iterations, {ARMS_TXT} (arm means per iteration) — the MI "
                                 f"target scored from the patient side. {GRADER}. {CENSOR}")
        plt.show()
    for m in PCT_METRICS:
        figm = plotting.single_behavior_trajectory(PCT_BEH, m, palette=S.PALETTE)
        if figm is not None:
            exports.save_fig(figm, m, group="pct",
                             caption=f"{eda_analysis.display_label(m)} across iterations, {ARMS_TXT} — per-signal "
                                     f"zoom of pct_detail_grid. {GRADER}. {CENSOR}")
            plt.close(figm)
    PT = PCT_BEH.round(3).sort_values(["arm", "iteration"]).reset_index(drop=True)
    display(PT)
    exports.save_table(PT, "pct_patient_by_iter",
                       caption="Mean PCT globals (1-5) + patient-utterance proportions per (arm, iteration), all four "
                               f"arms. {GRADER}. {CENSOR}")
    detail_deltas(PCT_BEH[["arm", "method", "K", "iteration"] + PCT_METRICS], "pct", "PCT",
                  xlabel="Δ vs base (globals 1–5 · proportions 0–1; % Sustain lower = better)",
                  sign_note="+ = higher value (globals / change-talk: better; % Sustain: worse).")

## 8 · MICI — MI-Inconsistency (severity + 6 behaviour rates; higher = worse)  `[EVAL]`
The negative-valence instrument — *which* harmful move drives an arm's MI-inconsistency rise? Severity global (1–5) + each MI-inconsistent behaviour **per therapist turn**, four arms overlaid. **Read:** compare which behaviour (over-praise / advise-without-permission / confront / warn / judge / persuade) carries the rise in each arm, and how the K=5 arm of each optimizer compares to its K=0 sibling. A positive Δ in the delta bars = **deterioration**. Per-behaviour zooms → group `mici/`.

In [ ]:
MICI_BEH = behavior.mici_behavior_by_iter(S.ARMS)
if MICI_BEH.empty:
    print("MICI not scored yet — run Run_Eval.ipynb with QUESTIONNAIRE_FILTER=['MICI'] to populate this.")
else:
    MICI_RATES = [c for c in MICI_BEH.columns if c.endswith("_rate")]  # the 6 per-behaviour rates
    MICI_METRICS = ["MICI_Severity", "MICI_Rate"] + MICI_RATES
    fig = plotting.behavior_trajectory_grid(
        MICI_BEH, palette=S.PALETTE, metrics=MICI_METRICS,
        title="MICI detail — severity + per-therapist-turn rates (higher = worse)")
    if fig:
        exports.save_fig(fig, "mici_detail_grid",
                         caption="MICI severity global (1-5) + total and per-behaviour MI-inconsistent rates PER THERAPIST "
                                 f"TURN across iterations, {ARMS_TXT} (arm means per iteration). HIGHER = WORSE. Isolates "
                                 f"WHICH harmful move (over-praise / advise-without-permission / confront / warn / judge / "
                                 f"persuade) carries each arm's MI-inconsistency rise. {GRADER}. {CENSOR}")
        plt.show()
    for m in ["MICI_Severity"] + MICI_RATES:
        figm = plotting.single_behavior_trajectory(MICI_BEH, m, palette=S.PALETTE)
        if figm is not None:
            exports.save_fig(figm, m, group="mici",
                             caption=f"{eda_analysis.display_label(m)} across iterations, {ARMS_TXT} — per-behaviour "
                                     f"zoom of mici_detail_grid (higher = worse). {GRADER}. {CENSOR}")
            plt.close(figm)
    MT = MICI_BEH.round(3).sort_values(["arm", "iteration"]).reset_index(drop=True)
    display(MT)
    exports.save_table(MT, "mici_behavior_by_iter",
                       caption="Mean MICI severity (1-5) + total and per-behaviour MI-inconsistent rates per therapist "
                               f"turn, per (arm, iteration), all four arms; higher = worse. {GRADER}. {CENSOR}")
    detail_deltas(MICI_BEH[["arm", "method", "K", "iteration"] + MICI_METRICS], "mici", "MICI",
                  xlabel="Δ vs base (severity 1–5 · rates /turn) — HIGHER = WORSE, positive Δ = deterioration",
                  sign_note="HIGHER = WORSE, so + Delta = deterioration.")

## 9 · Artifact index
Drop captions whose artifact no longer exists, then refresh `results/arms/INDEX.md` + the root `results/INDEX.md` (every notebook ends with this — whichever ran last completes the map).

In [ ]:
print("pruned captions:", exports.prune_orphan_captions())
print("index ->", exports.build_index())